# 00 EDA выбранного category-run

Этот notebook открывает исследование дедупликации SKU. Здесь мы не обучаем модели и не строим matcher, а проверяем качество входного среза: есть ли нужные поля, насколько заполнены `brand`, weight/pack-признаки и title, какие примеры похожих SKU стоит учитывать дальше.

**Результат:** компактный EDA-отчёт с таблицами, графиками и выводами для следующего этапа `01_candidate_generation.ipynb`.

## Оглавление

- [0. Локальные настройки](#0-локальные-настройки)
- [1. Подготовка окружения](#1-подготовка-окружения)
- [2. Почтовый архив: доказательство запросов по MPStats/ecom](#2-почтовый-архив-доказательство-запросов-по-mpstatsecom)
- [3. Конфигурация EDA](#3-конфигурация-eda)
- [4. Подключение к кубу и выбор категории](#4-подключение-к-кубу-и-выбор-категории)
- [5. Загрузка выбранного project/category среза из DuckDB](#5-загрузка-выбранного-projectcategory-среза-из-duckdb)
- [6. Вспомогательные функции](#6-вспомогательные-функции)
- [7. Базовая статистика и пропуски](#7-базовая-статистика-и-пропуски)
- [8. Анализ заполненности brand](#8-анализ-заполненности-brand)
- [9. Распределения weight/pack-колонок](#9-распределения-weightpack-колонок)
- [10. Аномалии weight/pack](#10-аномалии-weightpack)
- [11. Длина title](#11-длина-title)
- [12. Частотный анализ токенов title](#12-частотный-анализ-токенов-title)
- [13. Матрицы: brand x pack и числовые связи](#13-матрицы-brand--pack-и-числовые-связи)
- [14. Примеры возможных near-duplicates без ML](#14-примеры-возможных-near-duplicates-без-ml)
- [15. Примеры похожих, но разных товаров без ML](#15-примеры-похожих-но-разных-товаров-без-ml)
- [16. Итоговые выводы](#16-итоговые-выводы)

## Как читать notebook

Сначала задаём category-run, путь к DuckDB и опциональный путь к PST/OST-архиву почты. Если `MY_RUN_MAIL_ARCHIVE_EDA = True`, notebook считает входящие запросы по MPStats/ecom-темам, затем читает нужный срез `mpstats_products`, проверяет признаки и заканчивает выводами. Все эвристики в конце служат только для ручного EDA-просмотра, а не для production-матчинга.


## 0. Локальные настройки

Меняйте значения в следующей code-ячейке перед запуском. `.env` для этого notebook не нужен: category-run, путь к DuckDB, настройки почтового архива и лимиты примеров задаются прямо здесь, чтобы запуск был воспроизводимым на защите.


In [ ]:
# === MY notebook settings ===
# Какой research-run анализировать: "sauces", "coconut_oil" или "soap".
MY_CATEGORY_RUN = 'sauces'

# Если None, notebook ищет mpstats.duckdb через настройки приложения/корень проекта.
MY_DUCKDB_PATH = None

# Лимиты только для читаемых примеров в EDA, не для загрузки данных.
MY_MAX_EXAMPLE_PAIRS = 15
MY_MAX_HARD_NEGATIVE_EXAMPLES = 10

# Вес выше этого значения считаем подозрительным в диагностических таблицах.
MY_MAX_REASONABLE_WEIGHT_KG = 40.0

# Опциональный EDA почтового архива: доказывает, что часто приходят запросы по MPStats/ecom.
# Оставьте False для обычного SKU EDA. Для анализа укажите путь к .pst/.ost и включите True.
MY_RUN_MAIL_ARCHIVE_EDA = False
MY_MAIL_ARCHIVE_PATH = None  # пример: "/Users/me/Downloads/archive.pst"
MY_MAIL_MAX_MESSAGES = None  # можно поставить 5000 для быстрого smoke-run
MY_MAIL_INCLUDE_BODY = True  # False считает только subject/sender/headers и работает быстрее
MY_MAIL_EXPORT_DIR = "artifacts/reports/mail_archive_eda"

# Группы ключевых слов. Target groups ниже считают письмо релевантным запросом.
MY_MAIL_TOPIC_PATTERNS = {
    "MPStats": [
        r"\bmp\s*stats?\b",
        r"\bmpstats\b",
        r"мп\s*статс",
        r"мпстатс",
        r"мп\s*статистик",
    ],
    "E-commerce": [
        r"\be[-\s]?com(?:merce)?\b",
        r"\becommerce\b",
        r"\bеком\b",
        r"электронн\w*\s+коммерц",
    ],
    "Marketplace": [
        r"маркетплейс",
        r"\bmarketplace\b",
        r"\bozon\b",
        r"\bwb\b",
        r"wildberries",
        r"яндекс\s*маркет",
        r"yandex\s*market",
    ],
    "Analytics context": [
        r"аналитик",
        r"статистик",
        r"выгрузк",
        r"отч[её]т",
        r"дашборд",
        r"продаж",
        r"\bsku\b",
        r"категори",
    ],
}
MY_MAIL_TARGET_TOPIC_GROUPS = ["MPStats", "E-commerce", "Marketplace"]


## 1. Подготовка окружения

Эта ячейка находит корень репозитория, подключает существующие модули проекта и настраивает pandas/визуализации. Для подключения к кубу используется `duckdb_connection(..., read_only=True)`, поэтому notebook не меняет локальную БД.


In [ ]:
from __future__ import annotations

from collections import Counter
from itertools import combinations
import math
from pathlib import Path
import re
import sys
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError as exc:
    raise ImportError(
        "Для визуального EDA нужны matplotlib и seaborn. "
        "Установите их в окружение ноутбука, например: "
        "python3 -m pip install matplotlib seaborn"
    ) from exc


def find_project_root(start: Path) -> Path:
    '''Ищет корень репозитория по файлам, которые уже есть в проекте.'''
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "mpstats_app").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти корень проекта: нет AGENTS.md и mpstats_app/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mpstats_app.config import AppSettings
from mpstats_app.utils import quote_duckdb_name
from pipeline.repositories.sql_repository import duckdb_connection, table_exists
from pipeline.services.sales_filter_service import DEFAULT_SALES_FILTER_GROUP_COLUMNS, DEFAULT_SALES_MIN_QUANTILE, DEFAULT_SALES_MIN_UNITS, filter_sales_by_quantile
from research.dedup import resolve_category_run, resolve_run_paths

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)
sns.set_theme(style="whitegrid", context="notebook")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## 2. Почтовый архив: доказательство запросов по MPStats/ecom

Опциональный блок для анализа Outlook PST/OST-архива через `pypff`/`libpff`. Он нужен как доказательная часть: посчитать, как часто во входящей почте встречаются запросы по MPStats, e-commerce и маркетплейсам, показать динамику по месяцам и долю тем пирогом.

Чтобы запустить:

1. Установите системную библиотеку на машине, где лежит архив: `sudo apt install python3-pypff pff-tools`.
2. В первой code-ячейке задайте `MY_MAIL_ARCHIVE_PATH` и переключите `MY_RUN_MAIL_ARCHIVE_EDA = True`.
3. Запустите notebook сверху вниз. В git сохраняется только код анализа, а CSV/графики создаются локально в `MY_MAIL_EXPORT_DIR`.

Блок не сохраняет тело писем в CSV: body используется только для поиска ключевых слов, чтобы не превращать notebook в архив приватной переписки.


In [ ]:
MAIL_ARCHIVE_PATH = Path(MY_MAIL_ARCHIVE_PATH).expanduser() if MY_MAIL_ARCHIVE_PATH else None
MAIL_EXPORT_DIR = (PROJECT_ROOT / MY_MAIL_EXPORT_DIR).resolve()

mail_df = pd.DataFrame()
matched_mail_df = pd.DataFrame()
mail_archive_metadata: dict[str, object] = {
    "enabled": bool(MY_RUN_MAIL_ARCHIVE_EDA),
    "archive_path": str(MAIL_ARCHIVE_PATH) if MAIL_ARCHIVE_PATH else None,
}

HTML_TAG_RE = re.compile(r"<[^>]+>")
EMAIL_RE = re.compile(r"[A-Z0-9._%+-]+@([A-Z0-9.-]+\.[A-Z]{2,})", flags=re.IGNORECASE)


def decode_pff_value(value: object) -> str:
    """Приводит строковые/bytes поля pypff к обычному тексту."""
    if value is None:
        return ""
    if isinstance(value, bytes):
        for encoding in ("utf-8", "utf-16", "cp1251", "latin1"):
            try:
                return value.decode(encoding).replace("\x00", " ").strip()
            except UnicodeDecodeError:
                continue
        return value.decode("utf-8", errors="ignore").replace("\x00", " ").strip()
    return str(value).replace("\x00", " ").strip()


def safe_pff_attr(item: object, attr_names: Iterable[str], default: object = None) -> object:
    """Читает первое доступное pypff-свойство и не падает на битых item."""
    for attr_name in attr_names:
        try:
            value = getattr(item, attr_name)
        except Exception:
            continue
        if callable(value):
            try:
                value = value()
            except Exception:
                continue
        if value not in (None, ""):
            return value
    return default


def first_text(item: object, attr_names: Iterable[str]) -> str:
    return decode_pff_value(safe_pff_attr(item, attr_names, default=""))


def clean_message_text(value: object) -> str:
    text = decode_pff_value(value)
    text = HTML_TAG_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_message_datetime(value: object) -> pd.Timestamp:
    if value in (None, ""):
        return pd.NaT
    parsed = pd.to_datetime(value, errors="coerce", utc=True)
    if pd.isna(parsed):
        return pd.NaT
    return parsed.tz_convert(None)


def iter_pff_collection(item: object, collection_attr: str, count_attr: str, getter_attr: str):
    collection = safe_pff_attr(item, [collection_attr])
    if collection is not None:
        try:
            yield from collection
            return
        except TypeError:
            pass
    count = safe_pff_attr(item, [count_attr], default=0) or 0
    try:
        getter = getattr(item, getter_attr)
    except Exception:
        getter = None
    if not callable(getter):
        return
    for index in range(int(count)):
        try:
            yield getter(index)
        except Exception:
            continue


def iter_pff_messages(folder: object, folder_path: tuple[str, ...] = ()): 
    folder_name = first_text(folder, ["name", "display_name"]) or "ROOT"
    current_path = (*folder_path, folder_name)
    for message in iter_pff_collection(folder, "sub_messages", "number_of_sub_messages", "get_sub_message"):
        yield " / ".join(current_path), message
    for subfolder in iter_pff_collection(folder, "sub_folders", "number_of_sub_folders", "get_sub_folder"):
        yield from iter_pff_messages(subfolder, current_path)


def compile_mail_patterns(topic_patterns: dict[str, list[str]]) -> dict[str, list[re.Pattern[str]]]:
    return {
        topic: [re.compile(pattern, flags=re.IGNORECASE | re.UNICODE) for pattern in patterns]
        for topic, patterns in topic_patterns.items()
    }


def match_mail_topics(text: str, compiled_patterns: dict[str, list[re.Pattern[str]]]) -> tuple[list[str], list[str]]:
    topics: list[str] = []
    keywords: list[str] = []
    for topic, patterns in compiled_patterns.items():
        topic_hits = []
        for pattern in patterns:
            topic_hits.extend(match.group(0) for match in pattern.finditer(text))
        if topic_hits:
            topics.append(topic)
            keywords.extend(sorted(set(hit.strip() for hit in topic_hits if hit.strip())))
    return topics, sorted(set(keywords))


def sender_domain(sender_email: str, headers: str, sender_name: str) -> str:
    for value in (sender_email, headers, sender_name):
        match = EMAIL_RE.search(value or "")
        if match:
            return match.group(1).lower()
    return ""


def extract_message_record(folder_path: str, message: object, compiled_patterns: dict[str, list[re.Pattern[str]]]) -> dict[str, object]:
    subject = first_text(message, ["subject", "name", "conversation_topic"])
    sender_name = first_text(message, ["sender_name", "sender", "sender_display_name"])
    sender_email = first_text(message, ["sender_email_address", "sender_email", "sender_smtp_address"])
    headers = first_text(message, ["transport_headers", "headers"])
    message_date = parse_message_datetime(
        safe_pff_attr(message, ["delivery_time", "client_submit_time", "creation_time", "modification_time"])
    )

    body = ""
    if MY_MAIL_INCLUDE_BODY:
        body = clean_message_text(safe_pff_attr(message, ["plain_text_body", "text_body", "body", "html_body"]))
    text_for_match = "\n".join(part for part in [subject, sender_name, sender_email, headers, body] if part)
    matched_topics, matched_keywords = match_mail_topics(text_for_match, compiled_patterns)
    target_topic_set = set(MY_MAIL_TARGET_TOPIC_GROUPS)

    return {
        "message_date": message_date,
        "folder_path": folder_path,
        "sender_name": sender_name,
        "sender_email": sender_email,
        "sender_domain": sender_domain(sender_email, headers, sender_name),
        "subject": subject,
        "matched_topics": matched_topics,
        "matched_keywords": matched_keywords,
        "has_target_topic": bool(target_topic_set.intersection(matched_topics)),
        "body_chars_scanned": len(body),
    }


if not MY_RUN_MAIL_ARCHIVE_EDA:
    display(Markdown(
        "Почтовый EDA выключен. Чтобы посчитать запросы по MPStats/ecom, "
        "укажите `MY_MAIL_ARCHIVE_PATH` в первой code-ячейке и поставьте "
        "`MY_RUN_MAIL_ARCHIVE_EDA = True`."
    ))
elif MAIL_ARCHIVE_PATH is None or not MAIL_ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        "PST/OST archive не найден. Укажите существующий путь в MY_MAIL_ARCHIVE_PATH."
    )
else:
    try:
        import pypff
    except ImportError as exc:
        raise ImportError(
            "Не найден pypff/libpff. На Debian/Ubuntu установите: "
            "sudo apt install python3-pypff pff-tools. "
            "Если удобнее сначала экспортировать архив, можно использовать pffexport/readpst, "
            "а этот блок оставить для прямого чтения .pst/.ost."
        ) from exc

    if hasattr(pypff, "check_file_signature") and not pypff.check_file_signature(str(MAIL_ARCHIVE_PATH)):
        raise ValueError(f"Файл не похож на PST/OST/PAB архив libpff: {MAIL_ARCHIVE_PATH}")

    compiled_mail_patterns = compile_mail_patterns(MY_MAIL_TOPIC_PATTERNS)
    pff_file = pypff.open(str(MAIL_ARCHIVE_PATH))
    mail_archive_metadata.update({
        "pypff_version": pypff.get_version() if hasattr(pypff, "get_version") else "unknown",
        "archive_size_bytes": safe_pff_attr(pff_file, ["size"], default=None),
        "content_type": safe_pff_attr(pff_file, ["content_type"], default=None),
        "encryption_type": safe_pff_attr(pff_file, ["encryption_type"], default=None),
    })
    try:
        root_folder = safe_pff_attr(pff_file, ["root_folder"])
        if root_folder is None:
            raise RuntimeError("В архиве не найден root_folder.")
        records = []
        max_messages = None if MY_MAIL_MAX_MESSAGES in (None, 0) else int(MY_MAIL_MAX_MESSAGES)
        for folder_path, message in iter_pff_messages(root_folder):
            records.append(extract_message_record(folder_path, message, compiled_mail_patterns))
            if max_messages is not None and len(records) >= max_messages:
                break
    finally:
        pff_file.close()

    mail_df = pd.DataFrame(records)
    if not mail_df.empty:
        mail_df["message_date"] = pd.to_datetime(mail_df["message_date"], errors="coerce", utc=True).dt.tz_convert(None)
        matched_mail_df = mail_df.loc[mail_df["has_target_topic"]].copy()

    print(f"Всего прочитано писем: {len(mail_df):,}")
    print(f"Писем с MPStats/ecom/marketplace темами: {len(matched_mail_df):,}")
    display(pd.DataFrame([mail_archive_metadata]))


In [ ]:
if mail_df.empty:
    display(Markdown("Нет прочитанных писем для почтового EDA."))
else:
    matched_mail_df = mail_df.loc[mail_df["has_target_topic"]].copy()
    total_messages = len(mail_df)
    target_messages = len(matched_mail_df)
    target_share = target_messages / total_messages if total_messages else 0.0
    date_min = mail_df["message_date"].min()
    date_max = mail_df["message_date"].max()

    mail_summary = pd.DataFrame([
        {"metric": "Всего писем в срезе", "value": total_messages},
        {"metric": "Писем с MPStats/ecom/marketplace темами", "value": target_messages},
        {"metric": "Доля целевых писем", "value": f"{target_share:.1%}"},
        {"metric": "Уникальных доменов-отправителей в целевых письмах", "value": matched_mail_df["sender_domain"].replace("", np.nan).nunique()},
        {"metric": "Первое письмо", "value": date_min.date().isoformat() if pd.notna(date_min) else "нет даты"},
        {"metric": "Последнее письмо", "value": date_max.date().isoformat() if pd.notna(date_max) else "нет даты"},
    ])
    display(mail_summary)

    dated_all = mail_df.dropna(subset=["message_date"]).copy()
    dated_matched = matched_mail_df.dropna(subset=["message_date"]).copy()
    monthly_stats = pd.DataFrame()
    if not dated_all.empty:
        dated_all["month"] = dated_all["message_date"].dt.to_period("M").dt.to_timestamp()
        dated_matched["month"] = dated_matched["message_date"].dt.to_period("M").dt.to_timestamp()
        all_monthly = dated_all.groupby("month").size().rename("all_messages")
        target_monthly = dated_matched.groupby("month").size().rename("target_requests")
        monthly_stats = pd.concat([all_monthly, target_monthly], axis=1).fillna(0).astype(int)
        month_index = pd.date_range(monthly_stats.index.min(), monthly_stats.index.max(), freq="MS")
        monthly_stats = monthly_stats.reindex(month_index, fill_value=0).rename_axis("month").reset_index()
        monthly_stats["target_share"] = np.where(
            monthly_stats["all_messages"] > 0,
            monthly_stats["target_requests"] / monthly_stats["all_messages"],
            0.0,
        )
        monthly_stats["target_requests_3m_avg"] = monthly_stats["target_requests"].rolling(3, min_periods=1).mean()
        display(monthly_stats.tail(24))

    def choose_primary_topic(topics: list[str]) -> str:
        for topic in [*MY_MAIL_TARGET_TOPIC_GROUPS, *MY_MAIL_TOPIC_PATTERNS.keys()]:
            if topic in topics:
                return topic
        return "Other"

    topic_counts = pd.DataFrame(columns=["primary_topic", "messages"])
    if not matched_mail_df.empty:
        matched_mail_df["primary_topic"] = matched_mail_df["matched_topics"].apply(choose_primary_topic)
        topic_counts = (
            matched_mail_df.groupby("primary_topic")
            .size()
            .rename("messages")
            .sort_values(ascending=False)
            .reset_index()
        )
        display(topic_counts)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    if not monthly_stats.empty:
        axes[0].bar(monthly_stats["month"], monthly_stats["target_requests"], width=24, alpha=0.55, label="target requests")
        axes[0].plot(monthly_stats["month"], monthly_stats["target_requests_3m_avg"], color="tab:red", marker="o", label="3-month avg")
        axes[0].set_title("Динамика запросов по MPStats/ecom")
        axes[0].set_xlabel("Месяц")
        axes[0].set_ylabel("Количество писем")
        axes[0].legend()
        axes[0].tick_params(axis="x", rotation=45)
    else:
        axes[0].axis("off")
        axes[0].text(0.5, 0.5, "Нет дат для динамики", ha="center", va="center")

    if not topic_counts.empty:
        axes[1].pie(
            topic_counts["messages"],
            labels=topic_counts["primary_topic"],
            autopct="%1.0f%%",
            startangle=90,
            counterclock=False,
        )
        axes[1].set_title("Доля целевых писем по основной теме")
    else:
        axes[1].axis("off")
        axes[1].text(0.5, 0.5, "Нет целевых писем", ha="center", va="center")
    plt.tight_layout()
    plt.show()

    top_senders = (
        matched_mail_df.assign(sender_domain=matched_mail_df["sender_domain"].replace("", "unknown"))
        .groupby("sender_domain")
        .size()
        .rename("target_messages")
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    display(top_senders)

    sample_columns = ["message_date", "sender_domain", "folder_path", "subject", "matched_topics", "matched_keywords"]
    display(
        matched_mail_df.sort_values("message_date", ascending=False, na_position="last")[sample_columns].head(30)
    )

    MAIL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    mail_summary.to_csv(MAIL_EXPORT_DIR / "mail_archive_summary.csv", index=False)
    if not monthly_stats.empty:
        monthly_stats.to_csv(MAIL_EXPORT_DIR / "mail_archive_monthly_stats.csv", index=False)
    if not topic_counts.empty:
        topic_counts.to_csv(MAIL_EXPORT_DIR / "mail_archive_topic_counts.csv", index=False)
    matched_mail_df[sample_columns].to_csv(MAIL_EXPORT_DIR / "mail_archive_matched_messages.csv", index=False)
    print(f"Почтовые EDA-таблицы сохранены: {MAIL_EXPORT_DIR}")


## 3. Конфигурация EDA

`TARGET_CATEGORY` берётся из `MY_CATEGORY_RUN` в первой code-ячейке. Ниже ноутбук сам выберет точное значение из `Категория`: сначала проверит алиасы, затем ближайшее строковое совпадение.


In [ ]:
CATEGORY_RUN = resolve_category_run(MY_CATEGORY_RUN)
RUN_PATHS = resolve_run_paths(PROJECT_ROOT, CATEGORY_RUN)
TARGET_CATEGORY = CATEGORY_RUN.display_name
CATEGORY_ALIASES = list(CATEGORY_RUN.category_aliases)
PROJECT_NAME = CATEGORY_RUN.project_name
PRODUCTS_TABLE = "mpstats_products"
SALES_MIN_QUANTILE = DEFAULT_SALES_MIN_QUANTILE
SALES_MIN_UNITS = DEFAULT_SALES_MIN_UNITS
SALES_FILTER_DESCRIPTION = (
    f"min sales {SALES_MIN_UNITS:g}"
    if SALES_MIN_QUANTILE is None
    else f"bottom quantile {SALES_MIN_QUANTILE:.0%}; min sales {SALES_MIN_UNITS:g}"
)

MAX_EXAMPLE_PAIRS = int(MY_MAX_EXAMPLE_PAIRS)
MAX_HARD_NEGATIVE_EXAMPLES = int(MY_MAX_HARD_NEGATIVE_EXAMPLES)
MAX_REASONABLE_WEIGHT_KG = float(MY_MAX_REASONABLE_WEIGHT_KG)


def resolve_duckdb_path(project_root: Path) -> Path:
    '''Возвращает путь к существующему DuckDB-кубу и не создаёт новую БД.'''
    explicit_path = Path(MY_DUCKDB_PATH).expanduser() if MY_DUCKDB_PATH else None
    settings_path = AppSettings.create(project_root=project_root).db_path
    candidates = [explicit_path, settings_path]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "DuckDB-куб не найден. Укажите MY_DUCKDB_PATH в первой code-ячейке "
        f"или положите базу по стандартному пути: {settings_path}"
    )


DB_PATH = resolve_duckdb_path(PROJECT_ROOT)
print(f"Category run: {CATEGORY_RUN.slug} — {TARGET_CATEGORY}; project: {PROJECT_NAME}")
print(f"DuckDB cube: {DB_PATH}")


## 4. Подключение к кубу и выбор категории

Здесь мы не запускаем загрузку и не читаем raw CSV. Данные берутся из таблицы `mpstats_products`, которую наполняет существующий pipeline/web-app.


In [ ]:

with duckdb_connection(DB_PATH, read_only=True) as con:
    if not table_exists(con, PRODUCTS_TABLE):
        raise RuntimeError(f"В кубе нет таблицы {PRODUCTS_TABLE!r}.")

    columns_df = con.execute(f"PRAGMA table_info({quote_duckdb_name(PRODUCTS_TABLE)})").fetchdf()
    column_names = set(columns_df["name"].astype(str))
    project_filter_sql = ""
    project_filter_params: list[str] = []
    if PROJECT_NAME:
        if "__project_name" not in column_names:
            raise RuntimeError("Для category-run нужен project-фильтр, но в кубе нет __project_name.")
        project_filter_sql = f"WHERE {quote_duckdb_name('__project_name')} = ?"
        project_filter_params.append(PROJECT_NAME)

    categories_df = con.execute(
        f'''
        SELECT
            CAST({quote_duckdb_name("Категория")} AS VARCHAR) AS category,
            COUNT(*) AS rows
        FROM {quote_duckdb_name(PRODUCTS_TABLE)}
        {project_filter_sql}
        GROUP BY 1
        ORDER BY rows DESC
        ''',
        project_filter_params,
    ).fetchdf()

print(f"Колонок в {PRODUCTS_TABLE}: {len(columns_df)}")
display(columns_df[["name", "type"]])

display(categories_df.head(30))


In [ ]:

def normalize_category_name(value: object) -> str:
    '''Нормализация только для выбора реального имени категории в кубе.'''
    return re.sub(r"\s+", " ", str(value or "").strip().casefold())


def choose_real_category(categories: pd.DataFrame, aliases: list[str]) -> str:
    '''Выбирает категорию из куба: точный alias, затем ближайшее совпадение по символам.'''
    available = categories["category"].dropna().astype(str).tolist()
    normalized_to_original = {normalize_category_name(value): value for value in available}

    for alias in aliases:
        normalized = normalize_category_name(alias)
        if normalized in normalized_to_original:
            return normalized_to_original[normalized]

    # Это не matching товаров, а fallback для имени категории, если в данных оно в единственном числе.
    import difflib

    target = normalize_category_name(aliases[0])
    normalized_values = list(normalized_to_original)
    closest = difflib.get_close_matches(target, normalized_values, n=1, cutoff=0.45)
    if closest:
        return normalized_to_original[closest[0]]
    raise ValueError(f"Не нашёл категорию, похожую на {aliases[0]!r}. Доступно: {available[:20]}")


REAL_CATEGORY = choose_real_category(categories_df, CATEGORY_ALIASES)
print(f"Запрошено: {TARGET_CATEGORY!r}")
print(f"Реальное значение category в кубе: {REAL_CATEGORY!r}")


## 5. Загрузка выбранного project/category среза из DuckDB

Берём только нужные для EDA колонки. Title в разных версиях pipeline может называться `Название` или храниться в `SKU`, поэтому ниже выбирается первый реально доступный title-alias.


In [ ]:

PREFERRED_COLUMNS = [
    "__project_name",
    "__year",
    "__month",
    "__marketplace_code",
    "__source_type",
    "Дата",
    "Маркетплейс",
    "Категория",
    "Артикул",
    "SKU",
    "Название",
    "Бренд",
    "Подкатегория",
    "Тип",
    "Вес, кг (ед.)",
    "Вес, кг",
    "Вес аномалия",
    "Вес причина",
    "Объем, кг",
    "Объём, кг",
    "Объем, т",
    "Объём, т",
    "Продажи, шт",
    "Средняя цена, руб",
    "Выручка, руб",
    "Цена за кг",
]

available_columns = columns_df["name"].astype(str).tolist()
selected_columns = [column for column in PREFERRED_COLUMNS if column in available_columns]
missing_preferred_columns = [column for column in PREFERRED_COLUMNS if column not in available_columns]
select_sql = ",\n            ".join(quote_duckdb_name(column) for column in selected_columns)
where_clauses = [f"lower(trim(CAST({quote_duckdb_name('Категория')} AS VARCHAR))) = lower(trim(?))"]
query_params: list[object] = [REAL_CATEGORY]
if PROJECT_NAME:
    if "__project_name" not in available_columns:
        raise RuntimeError("Для category-run нужен project-фильтр, но в кубе нет __project_name.")
    where_clauses.append(f"{quote_duckdb_name('__project_name')} = ?")
    query_params.append(PROJECT_NAME)
where_sql = " AND ".join(where_clauses)

with duckdb_connection(DB_PATH, read_only=True) as con:
    df = con.execute(
        f'''
        SELECT
            {select_sql}
        FROM {quote_duckdb_name(PRODUCTS_TABLE)}
        WHERE {where_sql}
        ''',
        query_params,
    ).fetchdf()

rows_before_sales_filter = len(df)
df = filter_sales_by_quantile(
    df,
    sales_column="Продажи, шт",
    quantile=SALES_MIN_QUANTILE,
    min_sales=SALES_MIN_UNITS,
    group_columns=DEFAULT_SALES_FILTER_GROUP_COLUMNS,
).reset_index(drop=True)

if df.empty:
    raise RuntimeError(f"Срез category-run {CATEGORY_RUN.slug!r} пустой после sales-фильтра: {PROJECT_NAME!r} / {REAL_CATEGORY!r}.")

print(f"Строк после sales-фильтра: {len(df):,} из {rows_before_sales_filter:,} ({SALES_FILTER_DESCRIPTION})")
print("Недоступные optional-колонки:", missing_preferred_columns)
display(df.head(10))


## 6. Вспомогательные функции

Эти функции нужны только для EDA: привести текст к виду, удобному для подсчётов, выбрать колонку title и аккуратно считать пропуски. Они не заменяют production-загрузчики и не переписывают weight-parser.


In [ ]:

def first_existing(columns: Iterable[str], frame: pd.DataFrame) -> str | None:
    '''Возвращает первое имя колонки из списка, которое есть в DataFrame.'''
    for column in columns:
        if column in frame.columns:
            return column
    return None


def clean_text_series(series: pd.Series) -> pd.Series:
    '''Строковая очистка для аналитики пропусков и токенов.'''
    return series.astype("string").fillna("").str.replace("\u00a0", " ", regex=False).str.strip()


def non_empty_mask(series: pd.Series) -> pd.Series:
    '''True там, где значение не пустое после trim.'''
    return clean_text_series(series).ne("")


def numeric_series(frame: pd.DataFrame, column: str) -> pd.Series:
    '''Безопасно приводит колонку к числу, поддерживая десятичную запятую.'''
    return pd.to_numeric(
        frame[column]
        .astype("string")
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def tokenize_title(value: object) -> list[str]:
    '''Простая токенизация title для частотного анализа, без ML и embeddings.'''
    text = str(value or "").casefold().replace("ё", "е")
    return re.findall(r"[a-zа-я0-9]+", text)


TITLE_COL = first_existing(["Название", "SKU"], df)
SKU_ID_COL = first_existing(["Артикул", "SKU"], df)
BRAND_COL = first_existing(["Бренд", "brand"], df)
CATEGORY_COL = "Категория"

if TITLE_COL is None:
    raise RuntimeError("Не найдена title-колонка: ожидалась `Название` или `SKU`.")

analysis_df = df.copy()
analysis_df["title_text"] = clean_text_series(analysis_df[TITLE_COL])
analysis_df["title_norm"] = analysis_df["title_text"].str.casefold().str.replace("ё", "е", regex=False)
analysis_df["title_char_len"] = analysis_df["title_text"].str.len()
analysis_df["title_tokens"] = analysis_df["title_text"].map(tokenize_title)
analysis_df["title_token_len"] = analysis_df["title_tokens"].map(len)
analysis_df["brand_norm"] = clean_text_series(analysis_df[BRAND_COL]).str.casefold() if BRAND_COL else ""

print(f"Title column: {TITLE_COL}")
print(f"SKU/id column: {SKU_ID_COL}")
print(f"Brand column: {BRAND_COL}")


## 7. Базовая статистика и пропуски

Сначала смотрим объём среза, число уникальных SKU/title и заполненность ключевых колонок. Heatmap ниже показывает долю пропусков: чем темнее ячейка, тем хуже заполненность.


In [ ]:

key_columns = [
    column
    for column in [SKU_ID_COL, TITLE_COL, BRAND_COL, "Категория", "Подкатегория", "Тип", "Вес, кг (ед.)", "Вес, кг", "Продажи, шт", "Выручка, руб"]
    if column and column in analysis_df.columns
]

basic_stats = {
    "rows": len(analysis_df),
    "unique_sku_or_id": analysis_df[SKU_ID_COL].nunique(dropna=True) if SKU_ID_COL else np.nan,
    "unique_title": analysis_df["title_norm"].replace("", pd.NA).nunique(dropna=True),
    "categories_in_slice": analysis_df[CATEGORY_COL].nunique(dropna=True),
    "projects": analysis_df["__project_name"].nunique(dropna=True) if "__project_name" in analysis_df else np.nan,
    "marketplaces": analysis_df["Маркетплейс"].nunique(dropna=True) if "Маркетплейс" in analysis_df else np.nan,
    "months": analysis_df[["__year", "__month"]].drop_duplicates().shape[0] if {"__year", "__month"}.issubset(analysis_df.columns) else np.nan,
}

display(pd.DataFrame([basic_stats]).T.rename(columns={0: "value"}))

missing_summary = pd.DataFrame(
    {
        "column": key_columns,
        "missing_rows": [int((~non_empty_mask(analysis_df[column])).sum()) for column in key_columns],
        "missing_share": [float((~non_empty_mask(analysis_df[column])).mean()) for column in key_columns],
    }
).sort_values("missing_share", ascending=False)

display(missing_summary)

plt.figure(figsize=(10, max(3, 0.35 * len(missing_summary))))
sns.heatmap(
    missing_summary.set_index("column")[["missing_share"]],
    annot=True,
    fmt=".1%",
    cmap="Reds",
    cbar_kws={"label": "Доля пропусков"},
)
plt.title("Пропуски в ключевых колонках")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 8. Анализ заполненности `brand`

Бренд полезен как сигнал, но архитектурно не должен быть жёстким стоп-правилом. Здесь проверяем заполненность, топ брендов и примеры строк без бренда.


In [ ]:

if BRAND_COL is None:
    display(Markdown("В срезе нет колонки brand/Бренд."))
else:
    brand_filled = analysis_df["brand_norm"].ne("")
    brand_stats = pd.DataFrame(
        [
            {"metric": "brand filled rows", "value": int(brand_filled.sum())},
            {"metric": "brand empty rows", "value": int((~brand_filled).sum())},
            {"metric": "brand filled share", "value": round(float(brand_filled.mean()), 4)},
            {"metric": "unique non-empty brands", "value": int(analysis_df.loc[brand_filled, "brand_norm"].nunique())},
        ]
    )
    display(brand_stats)

    top_brands = analysis_df.loc[brand_filled, "brand_norm"].value_counts().head(25).rename_axis("brand").reset_index(name="rows")
    display(top_brands)

    plt.figure(figsize=(11, 7))
    sns.barplot(data=top_brands, y="brand", x="rows", color="#4C78A8")
    plt.title("Топ брендов по числу строк")
    plt.xlabel("Строк")
    plt.ylabel("Бренд")
    plt.tight_layout()
    plt.show()

    empty_brand_examples = analysis_df.loc[
        ~brand_filled,
        [column for column in [SKU_ID_COL, TITLE_COL, "Категория", "Подкатегория", "Вес, кг"] if column in analysis_df.columns],
    ].head(15)
    display(Markdown("### Примеры строк без brand"))
    display(empty_brand_examples)


## 9. Распределения weight/pack-колонок

Используем готовые колонки из текущего pipeline. В этом кубе ожидаем прежде всего `Вес, кг (ед.)` и `Вес, кг`; если появятся дополнительные pack-колонки, блок автоматически покажет доступные из списка ниже.


In [ ]:

PACK_NUMERIC_CANDIDATES = [
    "Вес, кг (ед.)",
    "Вес, кг",
    "Объем, кг",
    "Объём, кг",
    "Объем, т",
    "Объём, т",
]
pack_numeric_cols = [column for column in PACK_NUMERIC_CANDIDATES if column in analysis_df.columns]

for column in pack_numeric_cols:
    analysis_df[f"{column}__num"] = numeric_series(analysis_df, column)

if {"Вес, кг (ед.)", "Вес, кг"}.issubset(analysis_df.columns):
    unit_weight = analysis_df["Вес, кг (ед.)__num"]
    total_weight = analysis_df["Вес, кг__num"]
    analysis_df["pack_count_estimate"] = np.where(
        (unit_weight > 0) & (total_weight > 0),
        total_weight / unit_weight,
        np.nan,
    )
    pack_numeric_cols = [*pack_numeric_cols, "pack_count_estimate"]

pack_summary_rows = []
for column in pack_numeric_cols:
    series = analysis_df[f"{column}__num"] if f"{column}__num" in analysis_df else analysis_df[column]
    series = pd.to_numeric(series, errors="coerce")
    pack_summary_rows.append(
        {
            "column": column,
            "non_null": int(series.notna().sum()),
            "missing_share": round(float(series.isna().mean()), 4),
            "positive_share": round(float((series > 0).mean()), 4),
            "median": series.median(),
            "p95": series.quantile(0.95),
            "p99": series.quantile(0.99),
            "max": series.max(),
        }
    )

pack_summary = pd.DataFrame(pack_summary_rows)
display(pack_summary)

if pack_numeric_cols:
    n_cols = 2
    n_rows = math.ceil(len(pack_numeric_cols) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 4.2 * n_rows))
    axes = np.array(axes).reshape(-1)

    for ax, column in zip(axes, pack_numeric_cols):
        series = analysis_df[f"{column}__num"] if f"{column}__num" in analysis_df else analysis_df[column]
        values = pd.to_numeric(series, errors="coerce")
        values = values[(values > 0) & values.notna()]
        if values.empty:
            ax.set_title(f"{column}: нет положительных значений")
            ax.axis("off")
            continue
        upper = values.quantile(0.99)
        sns.histplot(values.clip(upper=upper), bins=40, ax=ax, color="#59A14F")
        ax.axvline(values.median(), color="#E15759", linestyle="--", label="median")
        ax.set_title(f"{column}: распределение до p99")
        ax.set_xlabel(column)
        ax.legend()

    for ax in axes[len(pack_numeric_cols):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("В срезе нет числовых weight/pack-колонок из ожидаемого списка."))


## 10. Аномалии weight/pack

Проверяем только уже готовые признаки. Это не новый parser: мы не извлекаем вес из title, а ищем подозрительные значения в колонках, которые уже создал pipeline.


In [ ]:

anomaly_flags = pd.DataFrame(index=analysis_df.index)

if "Вес, кг (ед.)__num" in analysis_df:
    anomaly_flags["unit_weight_missing_or_nonpositive"] = analysis_df["Вес, кг (ед.)__num"].isna() | (analysis_df["Вес, кг (ед.)__num"] <= 0)
    anomaly_flags["unit_weight_too_large"] = analysis_df["Вес, кг (ед.)__num"] > MAX_REASONABLE_WEIGHT_KG

if "Вес, кг__num" in analysis_df:
    anomaly_flags["total_weight_missing_or_nonpositive"] = analysis_df["Вес, кг__num"].isna() | (analysis_df["Вес, кг__num"] <= 0)
    anomaly_flags["total_weight_too_large"] = analysis_df["Вес, кг__num"] > MAX_REASONABLE_WEIGHT_KG

if {"Вес, кг (ед.)__num", "Вес, кг__num"}.issubset(analysis_df.columns):
    anomaly_flags["unit_weight_gt_total_weight"] = analysis_df["Вес, кг (ед.)__num"] > analysis_df["Вес, кг__num"] + 1e-9

if "pack_count_estimate" in analysis_df:
    anomaly_flags["pack_count_estimate_too_large"] = analysis_df["pack_count_estimate"] > 24
    anomaly_flags["pack_count_estimate_fractional"] = (
        analysis_df["pack_count_estimate"].notna()
        & (analysis_df["pack_count_estimate"] > 1.05)
        & ((analysis_df["pack_count_estimate"] - analysis_df["pack_count_estimate"].round()).abs() > 0.05)
    )

if "Вес аномалия" in analysis_df.columns:
    anomaly_flags["parser_weight_anomaly_flag"] = analysis_df["Вес аномалия"].astype("string").str.casefold().isin(["true", "1", "yes", "да"])

if anomaly_flags.empty:
    display(Markdown("Не нашлось доступных weight/pack-колонок для проверки аномалий."))
else:
    anomaly_summary = anomaly_flags.sum().sort_values(ascending=False).rename_axis("anomaly").reset_index(name="rows")
    anomaly_summary["share"] = anomaly_summary["rows"] / len(analysis_df)
    display(anomaly_summary)

    plt.figure(figsize=(10, max(3, 0.45 * len(anomaly_summary))))
    sns.barplot(data=anomaly_summary, y="anomaly", x="share", color="#F28E2B")
    plt.gca().xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    plt.title("Доля строк с weight/pack-аномалиями")
    plt.xlabel("Доля строк")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

    any_anomaly = anomaly_flags.any(axis=1)
    anomaly_columns_for_view = [
        column
        for column in [SKU_ID_COL, TITLE_COL, BRAND_COL, "Подкатегория", "Вес, кг (ед.)", "Вес, кг", "pack_count_estimate"]
        if column in analysis_df.columns
    ]
    display(Markdown("### Примеры строк с подозрительным weight/pack"))
    display(analysis_df.loc[any_anomaly, anomaly_columns_for_view].head(25))


## 11. Длина title

Для candidate generation важно понимать, насколько title информативен: слишком короткие названия дают мало сигнала, слишком длинные часто содержат маркетинговый шум.


In [ ]:

title_length_summary = analysis_df[["title_char_len", "title_token_len"]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
display(title_length_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(analysis_df["title_char_len"], bins=50, ax=axes[0], color="#4C78A8")
axes[0].axvline(analysis_df["title_char_len"].median(), color="#E15759", linestyle="--", label="median")
axes[0].set_title("Длина title в символах")
axes[0].set_xlabel("Символов")
axes[0].legend()

sns.histplot(analysis_df["title_token_len"], bins=40, ax=axes[1], color="#B07AA1")
axes[1].axvline(analysis_df["title_token_len"].median(), color="#E15759", linestyle="--", label="median")
axes[1].set_title("Длина title в токенах")
axes[1].set_xlabel("Токенов")
axes[1].legend()
plt.tight_layout()
plt.show()

short_titles = analysis_df.nsmallest(15, "title_token_len")[[column for column in [SKU_ID_COL, TITLE_COL, BRAND_COL, "Вес, кг"] if column in analysis_df.columns]]
long_titles = analysis_df.nlargest(15, "title_token_len")[[column for column in [SKU_ID_COL, TITLE_COL, BRAND_COL, "Вес, кг"] if column in analysis_df.columns]]

display(Markdown("### Самые короткие title"))
display(short_titles)
display(Markdown("### Самые длинные title"))
display(long_titles)


## 12. Частотный анализ токенов title

Это простой EDA-подсчёт слов, не модель. Он помогает увидеть шумные слова, вкусы, типы продукта и потенциальные признаки для будущего candidate generation.


In [ ]:

STOPWORDS = {
    "и", "в", "во", "на", "с", "со", "для", "без", "из", "по", "к", "от", "до", "под", "над",
    "а", "или", "не", "за", "при", "the", "and", "with", "of", "шт", "г", "гр", "кг", "мл", "л",
    "x", "х", "уп", "упак", "пак", "набор", "соус", "соуса", "соусы",
}


def useful_tokens(tokens: list[str]) -> list[str]:
    '''Убирает короткие и очень служебные токены для частотного обзора.'''
    out = []
    for token in tokens:
        if token in STOPWORDS:
            continue
        if token.isdigit():
            continue
        if len(token) < 3:
            continue
        out.append(token)
    return out


analysis_df["useful_title_tokens"] = analysis_df["title_tokens"].map(useful_tokens)
all_tokens = [token for tokens in analysis_df["useful_title_tokens"] for token in tokens]
token_counts = pd.DataFrame(Counter(all_tokens).most_common(40), columns=["token", "rows_or_mentions"])
display(token_counts)

plt.figure(figsize=(11, 9))
sns.barplot(data=token_counts.head(30), y="token", x="rows_or_mentions", color="#4C78A8")
plt.title("Частые токены title после простой очистки")
plt.xlabel("Упоминаний")
plt.ylabel("Токен")
plt.tight_layout()
plt.show()


In [ ]:

def ngrams(tokens: list[str], n: int) -> list[str]:
    '''Строит n-граммы из уже очищенных токенов.'''
    if len(tokens) < n:
        return []
    return [" ".join(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


bigram_counts = Counter()
trigram_counts = Counter()
for tokens in analysis_df["useful_title_tokens"]:
    bigram_counts.update(ngrams(tokens, 2))
    trigram_counts.update(ngrams(tokens, 3))

bigrams_df = pd.DataFrame(bigram_counts.most_common(25), columns=["bigram", "mentions"])
trigrams_df = pd.DataFrame(trigram_counts.most_common(25), columns=["trigram", "mentions"])

display(Markdown("### Частые биграммы"))
display(bigrams_df)
display(Markdown("### Частые триграммы"))
display(trigrams_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sns.barplot(data=bigrams_df.head(20), y="bigram", x="mentions", ax=axes[0], color="#59A14F")
axes[0].set_title("Топ биграмм")
axes[0].set_xlabel("Упоминаний")
axes[0].set_ylabel("")
sns.barplot(data=trigrams_df.head(20), y="trigram", x="mentions", ax=axes[1], color="#F28E2B")
axes[1].set_title("Топ триграмм")
axes[1].set_xlabel("Упоминаний")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:

DOMAIN_HINTS = [
    "чесночный", "чеснок", "сырный", "барбекю", "bbq", "терияки", "острый", "сладкий", "кисло", "чили",
    "томатный", "песто", "васаби", "соевый", "майонезный", "грибной", "сливочный", "ореховый", "перечный",
    "бальзамический", "устричный", "рыбный", "ткемали", "сацебели", "аджика", "кетчуп", "паста",
]

hint_rows = []
for hint in DOMAIN_HINTS:
    mask = analysis_df["title_norm"].str.contains(re.escape(hint), na=False)
    hint_rows.append({"hint": hint, "rows": int(mask.sum()), "share": float(mask.mean())})

hints_df = pd.DataFrame(hint_rows).sort_values("rows", ascending=False)
display(hints_df)

plt.figure(figsize=(10, 7))
sns.barplot(data=hints_df[hints_df["rows"] > 0].head(25), y="hint", x="rows", color="#B07AA1")
plt.title("Слова, похожие на вкус или тип продукта")
plt.xlabel("Строк")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 13. Матрицы: brand × pack и числовые связи

Матрица top-brand × округлённый вес помогает увидеть, где много одинаковых фасовок. Корреляционная heatmap полезна только как sanity-check числовых полей: она не доказывает дубли, но показывает странные связи и выбросы.


In [ ]:

if BRAND_COL and "Вес, кг__num" in analysis_df.columns:
    top_brand_names = analysis_df.loc[analysis_df["brand_norm"].ne(""), "brand_norm"].value_counts().head(15).index
    brand_pack_df = analysis_df.loc[analysis_df["brand_norm"].isin(top_brand_names)].copy()
    brand_pack_df["total_weight_rounded"] = brand_pack_df["Вес, кг__num"].round(2)
    top_weights = brand_pack_df["total_weight_rounded"].value_counts().head(20).index
    matrix = pd.crosstab(
        brand_pack_df.loc[brand_pack_df["total_weight_rounded"].isin(top_weights), "brand_norm"],
        brand_pack_df.loc[brand_pack_df["total_weight_rounded"].isin(top_weights), "total_weight_rounded"],
    )
    plt.figure(figsize=(14, 8))
    sns.heatmap(matrix, cmap="Blues", linewidths=0.2)
    plt.title("Матрица: топ бренды × округлённый общий вес")
    plt.xlabel("Вес, кг")
    plt.ylabel("Бренд")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("Матрица brand × pack пропущена: нет brand или общего веса."))

numeric_for_corr = []
for column in ["Вес, кг (ед.)__num", "Вес, кг__num", "pack_count_estimate", "Продажи, шт", "Средняя цена, руб", "Выручка, руб", "Цена за кг"]:
    if column in analysis_df.columns:
        numeric_for_corr.append(column)
    elif column in df.columns:
        analysis_df[f"{column}__num"] = numeric_series(analysis_df, column)
        numeric_for_corr.append(f"{column}__num")

corr_frame = analysis_df[numeric_for_corr].apply(pd.to_numeric, errors="coerce") if numeric_for_corr else pd.DataFrame()
if corr_frame.shape[1] >= 2:
    corr = corr_frame.corr(method="spearman")
    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
    plt.title("Spearman correlation по числовым колонкам")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("Недостаточно числовых колонок для корреляционной матрицы."))


## 14. Примеры возможных near-duplicates без ML

Это не matcher и не candidate generation. Блок только выбирает небольшие примеры для ручного просмотра: одинаковый нормализованный title и, если доступно, совпадающий округлённый вес/brand.


In [ ]:

def compact_title_for_exact_review(value: object) -> str:
    '''Нормализация title для поиска точных повторов в EDA-примерах.'''
    text = str(value or "").casefold().replace("ё", "е")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^0-9a-zа-я]+", " ", text)
    return text.strip()


review_df = analysis_df.copy()
review_df["title_exact_review_key"] = review_df[TITLE_COL].map(compact_title_for_exact_review)
if "Вес, кг__num" in review_df.columns:
    review_df["total_weight_review_key"] = review_df["Вес, кг__num"].round(3)
else:
    review_df["total_weight_review_key"] = np.nan

near_duplicate_group_cols = ["title_exact_review_key"]
if BRAND_COL:
    near_duplicate_group_cols.append("brand_norm")
if "total_weight_review_key" in review_df.columns:
    near_duplicate_group_cols.append("total_weight_review_key")

pairs = []
view_columns = [column for column in [SKU_ID_COL, TITLE_COL, BRAND_COL, "Подкатегория", "Вес, кг (ед.)", "Вес, кг", "Маркетплейс", "__project_name"] if column in review_df.columns]
for _, group in review_df[review_df["title_exact_review_key"].ne("")].groupby(near_duplicate_group_cols, dropna=False):
    if len(group) < 2:
        continue
    sample = group.sort_values(view_columns[0] if view_columns else TITLE_COL).head(3)
    for left_idx, right_idx in combinations(sample.index[:3], 2):
        left = review_df.loc[left_idx]
        right = review_df.loc[right_idx]
        pairs.append(
            {
                "reason": "same normalized title/brand/weight review key",
                "id_a": left.get(SKU_ID_COL),
                "title_a": left.get(TITLE_COL),
                "brand_a": left.get(BRAND_COL) if BRAND_COL else None,
                "weight_a": left.get("Вес, кг"),
                "id_b": right.get(SKU_ID_COL),
                "title_b": right.get(TITLE_COL),
                "brand_b": right.get(BRAND_COL) if BRAND_COL else None,
                "weight_b": right.get("Вес, кг"),
            }
        )
        if len(pairs) >= MAX_EXAMPLE_PAIRS:
            break
    if len(pairs) >= MAX_EXAMPLE_PAIRS:
        break

near_duplicates_df = pd.DataFrame(pairs)
if near_duplicates_df.empty:
    display(Markdown("Точных повторов по грубому review-key не найдено. Это нормально: на следующем этапе понадобится отдельная candidate generation."))
else:
    display(near_duplicates_df)


## 15. Примеры похожих, но разных товаров без ML

Здесь ищем hard negatives для будущей разметки: один бренд и близкая/одинаковая фасовка, но разные вкусы/типы или разные подкатегории. Это ручной EDA-сэмпл, не алгоритм матчинга.


In [ ]:

HARD_NEGATIVE_HINTS = {
    "чесночный", "чеснок", "сырный", "барбекю", "bbq", "терияки", "острый", "сладкий", "чили", "томатный",
    "песто", "васаби", "соевый", "грибной", "сливочный", "ореховый", "перечный", "кетчуп", "паста", "аджика",
}


def hint_set(tokens: list[str]) -> set[str]:
    '''Выделяет только заранее заданные вкусовые/типовые слова для ручного hard-negative обзора.'''
    return set(tokens) & HARD_NEGATIVE_HINTS


review_df["hint_set"] = review_df["useful_title_tokens"].map(hint_set) if "useful_title_tokens" in review_df else analysis_df["useful_title_tokens"].map(hint_set)
review_df["weight_bucket"] = review_df["Вес, кг__num"].round(2) if "Вес, кг__num" in review_df else np.nan

hard_negative_pairs = []
if BRAND_COL and "weight_bucket" in review_df.columns:
    grouped = review_df[
        review_df["brand_norm"].ne("")
        & review_df["weight_bucket"].notna()
        & review_df["title_exact_review_key"].ne("")
    ].groupby(["brand_norm", "weight_bucket"], dropna=False)

    for _, group in grouped:
        if len(group) < 2 or len(group) > 80:
            continue
        rows = group.head(30)
        for left_idx, right_idx in combinations(rows.index, 2):
            left = review_df.loc[left_idx]
            right = review_df.loc[right_idx]
            if left[TITLE_COL] == right[TITLE_COL]:
                continue
            different_subcategory = "Подкатегория" in review_df.columns and str(left.get("Подкатегория")) != str(right.get("Подкатегория"))
            different_hints = bool(left["hint_set"] or right["hint_set"]) and left["hint_set"] != right["hint_set"]
            if not (different_subcategory or different_hints):
                continue
            hard_negative_pairs.append(
                {
                    "reason": "same brand/weight, but different hints or subcategory",
                    "id_a": left.get(SKU_ID_COL),
                    "title_a": left.get(TITLE_COL),
                    "subcategory_a": left.get("Подкатегория"),
                    "hints_a": ", ".join(sorted(left["hint_set"])),
                    "id_b": right.get(SKU_ID_COL),
                    "title_b": right.get(TITLE_COL),
                    "subcategory_b": right.get("Подкатегория"),
                    "hints_b": ", ".join(sorted(right["hint_set"])),
                    "brand": left.get(BRAND_COL) if BRAND_COL else None,
                    "weight": left.get("Вес, кг"),
                }
            )
            if len(hard_negative_pairs) >= MAX_HARD_NEGATIVE_EXAMPLES:
                break
        if len(hard_negative_pairs) >= MAX_HARD_NEGATIVE_EXAMPLES:
            break

hard_negatives_df = pd.DataFrame(hard_negative_pairs)
if hard_negatives_df.empty:
    display(Markdown("Hard-negative примеры по простым правилам не найдены. Можно расширить ручной сэмпл после разметки."))
else:
    display(hard_negatives_df)


## 16. Итоговые выводы

Финальная code-ячейка собирает наблюдения из текущего запуска. Её удобно скопировать в отчёт или использовать как checklist перед candidate generation.


In [ ]:

rows_count = len(analysis_df)
unique_titles = int(analysis_df["title_norm"].replace("", pd.NA).nunique(dropna=True))
brand_fill_share = float(analysis_df["brand_norm"].ne("").mean()) if BRAND_COL else np.nan
empty_title_share = float(analysis_df["title_text"].eq("").mean())
median_title_tokens = float(analysis_df["title_token_len"].median())

weight_findings = []
if "Вес, кг__num" in analysis_df.columns:
    weight_missing_share = float((analysis_df["Вес, кг__num"].isna() | (analysis_df["Вес, кг__num"] <= 0)).mean())
    weight_findings.append(f"общий вес отсутствует или неположительный у {weight_missing_share:.1%} строк")
if "Вес, кг (ед.)__num" in analysis_df.columns:
    unit_weight_missing_share = float((analysis_df["Вес, кг (ед.)__num"].isna() | (analysis_df["Вес, кг (ед.)__num"] <= 0)).mean())
    weight_findings.append(f"вес единицы отсутствует или неположительный у {unit_weight_missing_share:.1%} строк")
if "pack_count_estimate" in analysis_df.columns:
    multipack_share = float((analysis_df["pack_count_estimate"] > 1.05).mean())
    weight_findings.append(f"оценочный multipack > 1 у {multipack_share:.1%} строк")

anomaly_text = "нет рассчитанных anomaly flags"
if "anomaly_summary" in globals() and not anomaly_summary.empty:
    top_anomalies = anomaly_summary[anomaly_summary["rows"] > 0].head(3)
    if not top_anomalies.empty:
        anomaly_text = "; ".join(f"{row.anomaly}: {row.share:.1%}" for row in top_anomalies.itertuples(index=False))
    else:
        anomaly_text = "явных weight/pack-аномалий по текущим правилам не найдено"

near_dup_text = f"найдено {len(near_duplicates_df)} ручных примеров возможных near-duplicates" if "near_duplicates_df" in globals() else "near-duplicate блок не выполнялся"
hard_negative_text = f"найдено {len(hard_negatives_df)} ручных hard-negative примеров" if "hard_negatives_df" in globals() else "hard-negative блок не выполнялся"

top_token_text = ", ".join(token_counts.head(10)["token"].tolist()) if "token_counts" in globals() and not token_counts.empty else "нет токенов"

summary_md = f'''
## Итоговые выводы

- В срезе `{REAL_CATEGORY}` загружено **{rows_count:,}** строк и **{unique_titles:,}** уникальных title.
- Пустые title: **{empty_title_share:.1%}**. Медианная длина title: **{median_title_tokens:.0f}** токенов.
- Заполненность brand: **{brand_fill_share:.1%}**. Brand выглядит полезным сигналом, но его нельзя делать жёстким стоп-гейтом: пустые и неоднозначные значения всё равно есть.
- Weight/pack: {"; ".join(weight_findings) if weight_findings else "в кубе нет ожидаемых weight/pack колонок"}.
- Главные weight/pack anomaly-сигналы: {anomaly_text}.
- Частые title-токены: {top_token_text}.
- Для ручного просмотра: {near_dup_text}; {hard_negative_text}.

### Что это значит для candidate generation

- Следующий этап нужно делать **внутри выбранной category**, как задано архитектурой.
- Candidate generation должен использовать title как главный сигнал, а brand и готовые weight/pack колонки — как дополнительные признаки, не как абсолютные запреты.
- Нужно отдельно учитывать hard negatives: похожий бренд и фасовка могут скрывать разные вкусы или типы продукта.
- До embeddings и моделей стоит зафиксировать простой baseline и разметочный файл с `exact_duplicate`, `different_product`, `uncertain`; pack/multipack разбираем после модели правилами.
'''

display(Markdown(summary_md))
